# ANÁLISIS DE LOS FACTORES ASOCIADOS A LA DESOCUPACIÓN EN JÓVENES DE 16 A 28 AÑOS EN BOLIVIA

**Programa:** Diplomado en Data Science — Versión I  
**Institución:** Universidad Mayor, Real y Pontificia de San Francisco Xavier de Chuquisaca (USFX)  
**Unidad de Posgrado:** Centro de Estudios de Posgrado e Investigación (CEPI)  
**Autor:** Gonzales Suyo Franz Reinaldo  
**Docente Coordinador:** Ing. Marcelo Arancibia  
**Sede y Gestión:** Sucre - Bolivia, 2026  

---

## Metodología CRISP-DM Adaptada
El presente cuaderno desarrolla de manera reproducible las fases analíticas del proyecto a partir de los microdatos de la **Encuesta Continua de Empleo (ECE) del 4to Trimestre de 2025** levantada por el Instituto Nacional de Estadística (INE).

### 1. Importación de Librerías y Configuración de Entorno

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Estilo de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Rutas del proyecto
BASE_DIR = Path.cwd()
DATA_RAW = BASE_DIR / "data" / "raw" / "ECE_4T2025.csv"
DATA_PROCESSED = BASE_DIR / "data" / "processed" / "ECE_4T2025_Jovenes_PEA.csv"
OUTPUTS_FIG = BASE_DIR / "outputs" / "figures"
OUTPUTS_TAB = BASE_DIR / "outputs" / "tables"

print(f"Directorio de trabajo: {BASE_DIR}")
print(f"Existe archivo fuente crudo: {DATA_RAW.exists()}")

### 2. Carga de Microdatos e Inspección Inicial (ECE 4T-2025)

In [ ]:
# Cargar microdatos (separador ';' y codificación latin1)
df_raw = pd.read_csv(DATA_RAW, sep=';', encoding='latin1', low_memory=False)
print(f"Dimensiones iniciales del dataset: {df_raw.shape[0]:,} registros x {df_raw.shape[1]} columnas")

### 3. Filtrado de la Población Objetivo y Preparación de Variables
- **Área urbana:** `area == 1`
- **Rango etario juvenil (Ley N.° 342):** `16 <= s1_03a <= 28`
- **Condición de actividad:** `pea == 1`

In [ ]:
# Conversiones de tipo y limpieza de valores nulos/espacios
df_raw['area_num'] = pd.to_numeric(df_raw['area'], errors='coerce').fillna(0).astype(int)
df_raw['edad'] = pd.to_numeric(df_raw['s1_03a'], errors='coerce').fillna(0).astype(int)
df_raw['pea_val'] = pd.to_numeric(df_raw['pea'], errors='coerce').fillna(0).astype(int)

# Aplicar filtros
df_juvenil = df_raw[(df_raw['area_num'] == 1) & (df_raw['edad'] >= 16) & (df_raw['edad'] <= 28) & (df_raw['pea_val'] == 1)].copy()

# Factor de expansión (reemplazar coma decimal por punto)
df_juvenil['peso_trimestral'] = df_juvenil['fact_trim_act'].astype(str).str.replace(',', '.').astype(float)

# Variables de empleo y desocupación
df_juvenil['es_desocupado'] = pd.to_numeric(df_juvenil['pead'], errors='coerce').fillna(0).astype(int)
df_juvenil['es_ocupado'] = pd.to_numeric(df_juvenil['peao'], errors='coerce').fillna(0).astype(int)
df_juvenil['es_cesante'] = pd.to_numeric(df_juvenil['peadces'], errors='coerce').fillna(0).astype(int)
df_juvenil['es_aspirante'] = pd.to_numeric(df_juvenil['peadasp'], errors='coerce').fillna(0).astype(int)
df_juvenil['es_subocupado'] = pd.to_numeric(df_juvenil['psubocup'], errors='coerce').fillna(0).astype(int)

# Segmentación etaria (4 tramos)
df_juvenil['grupo_edad'] = pd.cut(
    df_juvenil['edad'],
    bins=[15, 17, 20, 24, 28],
    labels=["16 a 17 años", "18 a 20 años", "21 a 24 años", "25 a 28 años"]
)

# Sexo y Departamento
df_juvenil['sexo'] = pd.to_numeric(df_juvenil['s1_02'], errors='coerce').map({1: 'Hombre', 2: 'Mujer'})
depto_map = {1: 'Chuquisaca', 2: 'La Paz', 3: 'Cochabamba', 4: 'Oruro', 5: 'Potosí', 6: 'Tarija', 7: 'Santa Cruz', 8: 'Beni', 9: 'Pando'}
df_juvenil['departamento'] = pd.to_numeric(df_juvenil['depto'], errors='coerce').map(depto_map)

print(f"Muestra filtrada (n): {len(df_juvenil):,} jóvenes encuestados")
print(f"Población PEA juvenil estimada: {df_juvenil['peso_trimestral'].sum():,.0f} personas")

### 4. Estimación de Tasas Ponderadas y Validación frente al Boletín INE

In [ ]:
pea_total = df_juvenil['peso_trimestral'].sum()
ocup_total = (df_juvenil['es_ocupado'] * df_juvenil['peso_trimestral']).sum()
desoc_total = (df_juvenil['es_desocupado'] * df_juvenil['peso_trimestral']).sum()
suboc_total = (df_juvenil['es_subocupado'] * df_juvenil['peso_trimestral']).sum()

tasa_desocupacion = (desoc_total / pea_total) * 100
tasa_subocupacion = (suboc_total / ocup_total) * 100

print(f"Tasa de Desocupación Juvenil Ponderada: {tasa_desocupacion:.2f}% (Meta oficial: 3.7%)")
print(f"Tasa de Subocupación Juvenil Ponderada: {tasa_subocupacion:.2f}% (Meta oficial: 8.7%)")

### 5. Análisis Bivariado y Pruebas Estadísticas (Chi-cuadrado y V de Cramér)

In [ ]:
variables_eval = ['grupo_edad', 'sexo', 'departamento']
res_tests = []

for var in variables_eval:
    tabla_cont = pd.crosstab(df_juvenil[var], df_juvenil['es_desocupado'])
    chi2, p_val, dof, _ = chi2_contingency(tabla_cont)
    n = tabla_cont.values.sum()
    r, c = tabla_cont.shape
    v_cramer = np.sqrt(chi2 / (n * min(r - 1, c - 1)))
    res_tests.append({
        'Variable': var,
        'Muestra': n,
        'Chi2': round(chi2, 4),
        'p-valor': p_val,
        'V de Cramér': round(v_cramer, 4),
        'Significativo (p < 0.05)': p_val < 0.05
    })

pd.DataFrame(res_tests)